In [2]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================

import pandas as pd
import re
import string
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 2. Common Text Preprocessing Pipeline



In [3]:
# ==============================
# 2. TEXT PREPROCESSOR
# ==============================

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stopwords = set([
            "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
            "you", "your", "yours", "he", "him", "his", "she", "her",
            "it", "its", "they", "them", "their", "am", "is", "are",
            "was", "were", "be", "been", "being", "have", "has", "had",
            "do", "does", "did", "the", "a", "an", "and", "but", "if",
            "or", "because", "as", "until", "while", "of", "at", "by",
            "for", "with", "about", "against", "between", "into", "through",
            "during", "before", "after", "above", "below", "to", "from",
            "up", "down", "in", "out", "on", "off", "over", "under"
        ])

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned_X = []

        for text in X:
            if not isinstance(text, str):
                text = str(text)

            # Lowercase
            text = text.lower()

            # Remove punctuation
            text = text.translate(str.maketrans("", "", string.punctuation))

            # Remove extra spaces and line breaks
            text = re.sub(r"\s+", " ", text).strip()

            # Remove stopwords
            words = text.split()
            words = [word for word in words if word not in self.stopwords]

            cleaned_X.append(" ".join(words))

        return cleaned_X

In [4]:
# ==============================
# 3. CREATE COMMON PIPELINE
# ==============================

def create_bec_pipeline(model_classifier):
    pipeline = Pipeline([
        ("preprocessor", TextPreprocessor()),
        ("tfidf", TfidfVectorizer()),
        ("classifier", model_classifier)
    ])
    return pipeline

## 3. Pipeline Verification



In [5]:
# ============================== 
# 4. PIPELINE VERIFICATION WITH SAMPLE DATA
# ==============================

sample_data = [
    "URGENT: Please review the invoice and wire the money immediately",
    "Hey, are we still meeting for lunch today?"
]

sample_labels = [1, 0]

test_pipeline = create_bec_pipeline(GradientBoostingClassifier(random_state=42))

print("--- Testing pipeline fit ---")
test_pipeline.fit(sample_data, sample_labels)
print("Pipeline works successfully.\n")

print("--- Testing pipeline predict ---")
new_email = ["Please send money to this bank account right now"]
print("Prediction:", test_pipeline.predict(new_email))

--- Testing pipeline fit ---
Pipeline works successfully.

--- Testing pipeline predict ---
Prediction: [1]


## 4. Load Dataset and Apply Data Filtering



In [6]:
# ==============================
# 5. LOAD DATASET + FILTER DATA
# ==============================

clean_df = pd.read_csv("synthetic_emails.csv")
poisoned_df = pd.read_csv("synthetic_emails_poisoned.csv")

print("Original clean dataset shape:", clean_df.shape)
print("Original poisoned dataset shape:", poisoned_df.shape)

# Keep only email pairs whose body was actually modified
modified_indices = []

for i in range(len(clean_df)):
    clean_body = str(clean_df.loc[i, "body"])
    poisoned_body = str(poisoned_df.loc[i, "body"])

    if clean_body != poisoned_body:
        modified_indices.append(i)

clean_filtered = clean_df.loc[modified_indices].copy()
poisoned_filtered = poisoned_df.loc[modified_indices].copy()

print("Clean filtered shape:", clean_filtered.shape)
print("Poisoned filtered shape:", poisoned_filtered.shape)

# Create binary target labels
clean_filtered["target"] = 0       # 0 = Clean BEC
poisoned_filtered["target"] = 1    # 1 = Poisoned BEC

# Combine filtered datasets
df = pd.concat([clean_filtered, poisoned_filtered], ignore_index=True)

# Handle missing text values
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

# Combine subject and body into one text column
df["text"] = df["subject"] + " " + df["body"]

X = df["text"]
y = df["target"]

print("Final dataset shape:", df.shape)
print("Class distribution:")
print(y.value_counts())

Original clean dataset shape: (4211, 3)
Original poisoned dataset shape: (4211, 3)
Clean filtered shape: (1187, 3)
Poisoned filtered shape: (1187, 3)
Final dataset shape: (2374, 5)
Class distribution:
target
0    1187
1    1187
Name: count, dtype: int64


## 5. Train/Test Split



In [7]:
# ==============================
# 6. TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])
print("Test class distribution:")
print(y_test.value_counts())

Training set size: 1899
Testing set size: 475
Test class distribution:
target
0    238
1    237
Name: count, dtype: int64


## 6. Train Gradient Boosting Model



In [8]:
# ==============================
# 7. TRAIN GRADIENT BOOSTING MODEL
# ==============================

model = GradientBoostingClassifier(random_state=42)

pipeline = create_bec_pipeline(model)

pipeline.fit(X_train, y_train)

print("Gradient Boosting training completed.")

Gradient Boosting training completed.


## 7. Evaluation



In [9]:
# ==============================
# 8. EVALUATION FUNCTION
# ==============================

from matplotlib import cm


def evaluate_model(model_name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred) 
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"===== {model_name} Result =====")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Clean BEC", "Poisoned BEC"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

In [10]:
# ==============================
# 9. SHOW RESULT
# ==============================

gradient_boosting_result = evaluate_model(
    "Gradient Boosting",
    pipeline,
    X_test,
    y_test
)

gradient_boosting_result

===== Gradient Boosting Result =====
Accuracy: 0.8589473684210527
Precision: 0.8571428571428571
Recall: 0.8607594936708861
F1-score: 0.8589473684210527

Classification Report:
              precision    recall  f1-score   support

   Clean BEC       0.86      0.86      0.86       238
Poisoned BEC       0.86      0.86      0.86       237

    accuracy                           0.86       475
   macro avg       0.86      0.86      0.86       475
weighted avg       0.86      0.86      0.86       475


Confusion Matrix:
[[204  34]
 [ 33 204]]


{'Model': 'Gradient Boosting',
 'Accuracy': 0.8589473684210527,
 'Precision': 0.8571428571428571,
 'Recall': 0.8607594936708861,
 'F1-score': 0.8589473684210527}